In [ ]:
import torch
import torchaudio
import librosa
import soundfile as sf
import random
import io
from pydub import AudioSegment
from pydub.generators import WhiteNoise

In [ ]:
AUDIO_PATH = './speech.wav'
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

## Predictor / Transcriber

In [ ]:
def Transcriber (audiopath):
    # load
    waveform, sample_rate = torchaudio.load(audiopath)

    # resample
    if sample_rate != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sample_rate, bundle.sample_rate)

    # emission
    with torch.inference_mode():
        emission, _ = model(waveform)

    print(emission)

    # class
    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emmision: torch.Tensor):
            idx = torch.argmax(waveform, dim=-1)
            idx = torch.unique_consecutive(idx, dim=-1)
            idx = [i for i in idx if i != self.blank]
            return " ".join([self.labels[i] for i in idx])

    # transcribe
    decoder = GreedyCTCDecoder(bundle.get_labels)
    txt = decoder(emission[0])
    txt = txt.replace("|", " ")
    return txt

## Support Function

## Menu Function